# Combined MASD Report Generator
## Handles: Jalna, Ujjain, Meghalaya

In [9]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
import glob


# Hardcoded Paths

In [10]:
BASE_DIR = r"c:\Users\AayushmanSingh\Desktop\MASD COMBINED"
INPUT_DIR = os.path.join(BASE_DIR, "Inputs")
COLORCODED_DIR = os.path.join(BASE_DIR, "Colour coded of combined script")
COMBINED_DIR = os.path.join(BASE_DIR, "Combined of combined script")
ROLE_FILE = os.path.join(BASE_DIR, "Role and department sheet.xlsx")
LOCATION_UJJAIN = os.path.join(BASE_DIR, "location_ujjain.xlsx")
LOCATION_MEGHALAYA = os.path.join(BASE_DIR, "Location Meghalaya.xlsx")
LOCATION_JALNA = os.path.join(BASE_DIR, "location jalna.xlsx")
EXPECTED_DIR = os.path.join(BASE_DIR, "expected")

# Expected row count configurations for validation
EXPECTED_ROW_COUNTS = {
    "meghalaya": 759,
    "jalna": 216,
    "ujjain": 236
}


# Region Configurations

In [11]:
def get_matching_files(pattern):
    matched_files = sorted(glob.glob(os.path.join(INPUT_DIR, pattern)))
    if not matched_files:
        print(f"Warning: No files matched pattern: {pattern}")
    return matched_files

REGION_CONFIGS = {
    "jalna": {
        "name": "Jalna",
        "csv_files": get_matching_files("MH Jalna_*_MASD.csv"),
        "state_prefix": "MH",  # Matches MH Jalna filename prefix
        "district_key": "Jalna",
        "location_master": LOCATION_JALNA,  # Uses Jalna's location details
        "tac_threshold_staff_nurse": 3,
        "tac_threshold_other": 3,
        "adoption_cols": ["Antenatal care", "At time of delivery", "PNC LTE 5 Months", "PNC GTE 5 Months"],
        "apply_check_growth_bf_cf_rules": False,  # Jalna skips these color rules
        "has_region_column": False,
        "filter_undefined_location": False,
        "output_combined": os.path.join(COMBINED_DIR, "MASD_Jalna_160626_combined.xlsx"),
        "output_colorcoded": os.path.join(COLORCODED_DIR, "MASD_Jalna_160626_colorcoded.xlsx"),
        "location_master_district_col": "District",
        "expected_combined": os.path.join(EXPECTED_DIR, "MASD_Jalna_160626_combined_expected.xlsx"),
        "expected_colorcoded": os.path.join(EXPECTED_DIR, "MASD_Jalna_160626_colorcoded (2)_expected.xlsx"),
    },
    "ujjain": {
        "name": "Ujjain",
        "csv_files": get_matching_files("MP Ujjain_*_MASD.csv"),
        "state_prefix": "MP",
        "location_master": LOCATION_UJJAIN,
        "tac_threshold_staff_nurse": 10,
        "tac_threshold_other": 6,
        "adoption_cols": ["Antenatal care", "At time of delivery", "PNC LTE 5 Months", "PNC GTE 5 Months"],
        "apply_check_growth_bf_cf_rules": True,
        "has_region_column": False,
        "filter_undefined_location": False,
        "output_combined": os.path.join(COMBINED_DIR, "MASD_Ujjain_160626_combined.xlsx"),
        "output_colorcoded": os.path.join(COLORCODED_DIR, "MASD_Ujjain_160626_colorcoded.xlsx"),
        "location_master_district_col": "District",
        "expected_combined": os.path.join(EXPECTED_DIR, "MASD_Ujjain_160626_combined_expected.xlsx"),
        "expected_colorcoded": os.path.join(EXPECTED_DIR, "MASD_Ujjain_160626_colorcoded_expected.xlsx"),
    },
    "meghalaya": {
        "name": "Meghalaya",
        "csv_files": get_matching_files("ML *_MASD.csv"),
        "state_prefix": "ML",
        "location_master": LOCATION_MEGHALAYA,
        "tac_threshold_staff_nurse": 10,
        "tac_threshold_other": 6,
        "adoption_cols": ["First Trimester", "Second Trimester", "Third Trimester", "At time of delivery", "PNC LTE 5 Months", "PNC GTE 5 Months"],
        "apply_check_growth_bf_cf_rules": True,
        "has_region_column": True,
        "filter_undefined_location": True,
        "output_combined": os.path.join(COMBINED_DIR, "MASD_Meghalaya_160626_combined.xlsx"),
        "output_colorcoded": os.path.join(COLORCODED_DIR, "MASD_Meghalaya_160626_colorcoded.xlsx"),
        "location_master_district_col": "district",
        "expected_combined": os.path.join(EXPECTED_DIR, "MASD_Meghalaya_160626_combined_expected.xlsx"),
        "expected_colorcoded": os.path.join(EXPECTED_DIR, "MASD_Meghalaya_160626_colorcoded (2)_expected.xlsx"),
    },
}


# Utility Functions

In [12]:
def extract_district(file_name, state_prefix):
    """Extract full district name from file name."""
    base_name = os.path.splitext(os.path.basename(file_name))[0]
    first_part = base_name.split("_")[0]
    # For ML: replace "ML " (with space); for MP/MH: replace "MP"/"MH"
    if state_prefix == "ML":
        district = first_part.replace("ML ", "").strip()
    else:
        district = first_part.replace(state_prefix, "").strip()
    return district


def clean_location(col):
    """Clean location text for matching."""
    return (
        col.astype(str)
        .str.split(",")
        .str[0]
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"\s*-\s*", "-", regex=True)
    )


def process_location_columns_ujjain_jalna(df, master_path):
    """
    Standardizes location using master Excel file.
    Used by Ujjain and Jalna (same location master structure).
    Preserves the District from filename if master mapping fails.
    """
    location_master = pd.read_excel(master_path)

    df["Location_clean"] = clean_location(df["Location"])
    location_master["Location_clean"] = clean_location(location_master["Location"])

    location_master = location_master.drop_duplicates(subset=["Location_clean"])

    df = df.merge(
        location_master[["Location_clean", "State", "District", "Actual location", "Block/Taluk"]],
        on="Location_clean",
        how="left"
    )

    # Preserve filename District when master mapping fails
    if "District_x" in df.columns and "District_y" in df.columns:
        df["District"] = df["District_y"].fillna(df["District_x"])
        df.drop(columns=["District_x", "District_y"], inplace=True)
    elif "District" not in df.columns:
        df["District"] = ""

    unmatched_df = df[df["Actual location"].isna()].copy()
    if not unmatched_df.empty:
        print(f"Warning: {len(unmatched_df)} rows could not be matched with location master.")

    for col in ["Actual location", "Block/Taluk", "District"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .fillna("")
                .astype(str)
                .str.strip()
                .str.title()
            )

    if "Location_clean" in df.columns:
        df.drop(columns=["Location_clean"], inplace=True)

    return df


def process_location_columns_meghalaya(df, master_path):
    """
    Standardizes location using Meghalaya master Excel file.
    Note: Meghalaya master uses lowercase 'district' column.
    """
    location_master = pd.read_excel(master_path)

    df["Location_clean"] = clean_location(df["Location"])
    location_master["Location_clean"] = clean_location(location_master["Location"])

    location_master = location_master.drop_duplicates(subset=["Location_clean"])

    df = df.merge(
        location_master[
            ["Location_clean", "State", "district", "Actual location", "Block/Taluk"]
        ],
        on="Location_clean",
        how="left"
    )

    unmatched_df = df[df["Actual location"].isna()].copy()

    # Clean final formatting
    df["Actual location"] = df["Actual location"].str.title()
    df["Block/Taluk"] = df["Block/Taluk"].str.title()
    df["District"] = df["District"].str.title()

    df.drop(columns=["Location_clean"], inplace=True)

    return df


def create_batch_column(df):
    """Creates a 'Batch' column in format: District B#"""
    df["Batch_Number"] = (
        df["Training Batch"]
        .astype(str)
        .str.extract(r'Batch\s*(\d+)', expand=False)
    )
    df["Batch"] = None
    mask = df["Batch_Number"].notna()
    df.loc[mask, "Batch"] = (
        df.loc[mask, "District"].astype(str).str.strip()
        + " B"
        + df.loc[mask, "Batch_Number"]
    )
    df.drop(columns=["Batch_Number"], inplace=True)
    return df


def apply_role_mapping(df, role_file_path):
    """Merges role mapping file with main dataframe."""
    roles_map = pd.read_excel(role_file_path)
    
    # Append manual mapping for "Ward In Charge" (which is missing in Excel but present in Meghalaya expected output)
    ward_row = pd.DataFrame([{
        "role": "Ward In Charge",
        "Short role": "WardInc_SN",
        "Department": "HFW",
        "Role group": "Staff nurse"
    }])
    roles_map = pd.concat([roles_map, ward_row], ignore_index=True)
    
    roles_map["role_clean"] = roles_map["role"].astype(str).str.strip().str.lower()
    df["Role_clean"] = df["Role"].astype(str).str.strip().str.lower()

    df = df.merge(
        roles_map,
        left_on="Role_clean",
        right_on="role_clean",
        how="left"
    )

    df.rename(columns={
        "Short role": "Roles defined",
        "Role group": "Role group",
        "Department": "Department"
    }, inplace=True)

    df.drop(columns=["Role_clean", "role_clean", "role"], inplace=True, errors="ignore")

    # df = df[~(df['Role'].isin([
    #     'Pediatrician + Batch Monitor',
    #     'Lady Supervisor (AWS) + Batch Monitors',
    #     'Cuedwell Support',
    #     'HST Data Analysis Group',
    #     'HST Data Clerk'
    # ]))]

    return df


def calculate_total_adoptions(df, adoption_cols):
    """Calculates Total adoptions as sum of specified columns."""
    for col in adoption_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df["Total adoptions"] = df[adoption_cols].sum(axis=1)
    return df


def create_total_adoptions_group(df):
    """Creates 'Total adoptions group'"""
    def total_adop_group(value):
        if pd.isna(value) or value < 1:
            return "NO"
        elif 1 <= value < 4:
            return "01_to_03"
        elif 4 <= value < 7:
            return "04_to_06"
        else:
            return "07_or_more"

    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
    df["Total adoptions group"] = df["Total adoptions"].apply(total_adop_group).astype(str)
    return df


def create_total_adoptions_group_monitoring(df):
    """Creates 'Total adoptions group_monitoring'"""
    def adop_group(value):
        if pd.isna(value):
            return "NO"
        elif 0 <= value <= 2:
            return "00_to_02"
        elif value == 3:
            return "03"
        else:
            return "04_or_more"

    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
    df["Total adoptions group_monitoring"] = df["Total adoptions"].apply(adop_group).astype(str)
    return df


def calculate_overall_activities(df):
    """Calculates Overall activities."""
    cols = ["Antenatal care", "Mother's Protein Intake", "Check growth", "Check BF", "Check CF"]
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df["Overall activities"] = df[cols].sum(axis=1)
    return df


def create_overall_activities_group(df):
    """Creates 'Overall activities group'"""
    def overall_activity_category(x):
        if pd.isna(x) or x == 0:
            return "None"
        elif 1 <= x <= 9:
            return "01 to 09"
        elif 10 <= x <= 19:
            return "10 to 19"
        elif 20 <= x <= 29:
            return "20 to 29"
        elif 30 <= x <= 39:
            return "30 to 39"
        elif 40 <= x <= 49:
            return "40 to 49"
        elif 50 <= x <= 59:
            return "50 to 59"
        elif 60 <= x <= 69:
            return "60 to 69"
        elif 70 <= x <= 79:
            return "70 to 79"
        elif 80 <= x <= 89:
            return "80 to 89"
        elif 90 <= x <= 99:
            return "90 to 99"
        else:
            return "Hundred or more"

    df["Overall activities"] = pd.to_numeric(df["Overall activities"], errors="coerce")
    df["Overall activities group"] = df["Overall activities"].apply(overall_activity_category).astype(str)
    return df


def calculate_percentage_adopted(df, total_cases=10):
    """Calculates Percentage of cases adopted."""
    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce").fillna(0)
    if total_cases == 0:
        df["Percentage of cases adopted"] = 0
    else:
        df["Percentage of cases adopted"] = ((df["Total adoptions"] * 100) / total_cases).round(2)
    return df


def create_last_activity_group(df):
    """Creates Last activity group based on No Activity Days."""
    df["No Activity Days"] = pd.to_numeric(df["No Activity Days"], errors="coerce")
    df["Last activity group"] = pd.cut(
        df["No Activity Days"],
        bins=[-1, 3, 7, float("inf")],
        labels=["03 or less days", "04 to 07 days", "08 or more days"]
    ).astype(object)
    df.loc[df["No Activity Days"].isna(), "Last activity group"] = "No activity"
    return df


def create_last_activity_group_monitoring(df):
    """Creates 'Last activity group_monitoring'"""
    df["No Activity Days"] = pd.to_numeric(df["No Activity Days"], errors="coerce")
    df["Last activity group_monitoring"] = pd.cut(
        df["No Activity Days"],
        bins=[-1, 3, 7, float("inf")],
        labels=["03 or less days", "04 to 07 days", "08 or more days"]
    ).astype(str)
    df.loc[df["No Activity Days"].isna(), "Last activity group_monitoring"] = "No activity"
    return df


def create_case_categories(df):
    """Creates category columns for multiple case-related fields."""
    bins = [-1, 0, 3, 6, 9, float("inf")]
    labels = ["No", "01 to 03", "04 to 06", "07 to 09", "10 or more"]

    category_columns = {
        "Antenatal Care category": "Antenatal care",
        "First Trimester category": "First Trimester",
        "Second Trimester category": "Second Trimester",
        "Third Trimester category": "Third Trimester",
        "Time of Delivery category": "At time of delivery",
        "PNC LTE 5 Months category": "PNC LTE 5 Months",
        "PNC GTE 5 Months category": "PNC GTE 5 Months",
        "Antenatal cases category": "Antenatal care",
        "Total cases category": "Total Active Cases",
        "Child cases category": "Total Child Cases"
    }

    for new_col, base_col in category_columns.items():
        df[base_col] = pd.to_numeric(df[base_col], errors="coerce")
        df[new_col] = pd.cut(df[base_col], bins=bins, labels=labels).astype(str)

    return df


def create_growth_measurement_category(df):
    """Creates 'Growth Measurement category' based on 'Check growth'"""
    bins = [-1, 0, 9, 19, 29, 39, 49, float("inf")]
    labels = ["None", "01 to 09", "10 to 19", "20 to 29", "30 to 39", "40 to 49", "50 or more"]
    df["Check growth"] = pd.to_numeric(df["Check growth"], errors="coerce")
    df["Growth Measurement category"] = pd.cut(df["Check growth"], bins=bins, labels=labels).astype(str)
    return df


def create_assess_bf_category(df):
    """Creates 'Assess BF category' based on 'Check BF'"""
    bins = [-1, 0, 2, 4, 6, 8, 10, float("inf")]
    labels = ["None", "01 to 02", "03 to 04", "05 to 06", "07 to 08", "09 to 10", "11 or more"]
    df["Check BF"] = pd.to_numeric(df["Check BF"], errors="coerce")
    df["Assess BF category"] = pd.cut(df["Check BF"], bins=bins, labels=labels).astype(str)
    return df


def create_assess_cf_category(df):
    """Creates 'Assess CF category' based on 'Check CF'"""
    bins = [-1, 0, 4, 8, 12, float("inf")]
    labels = ["None", "01 to 04", "05 to 08", "09 to 12", "13 or more"]
    df["Check CF"] = pd.to_numeric(df["Check CF"], errors="coerce")
    df["Assess CF category"] = pd.cut(df["Check CF"], bins=bins, labels=labels).astype(str)
    return df


def create_adoption_group_skill_verify(df):
    """Creates 'Adoption group for skill verify' based on 'Total adoptions'"""
    bins = [-1, 0, 5, float("inf")]
    labels = ["None", "01 to 05 adoptions", "06 or more adoption"]
    df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
    df["Adoption group for skill verify"] = pd.cut(df["Total adoptions"], bins=bins, labels=labels).astype(str)
    return df


def create_avg_activities_per_adoption(df):
    """Creates 'Avg activities/adoption' column"""
    df["Overall activities"] = pd.to_numeric(df["Overall activities"], errors="coerce")
    df["Total Active Cases"] = pd.to_numeric(df["Total Active Cases"], errors="coerce")
    df["Avg activities/adoption"] = (df["Overall activities"] / df["Total Active Cases"])
    df["Avg activities/adoption"] = df["Avg activities/adoption"].replace([np.inf, -np.inf], np.nan)
    df["Avg activities/adoption"] = df["Avg activities/adoption"].round(2)
    return df


def add_place_of_intervention(df, region_name):
    """Place of intervention = Facility if Staff Nurse, else Community"""
    # Jalna checks "Staff Nurse" (uppercase N) due to a bug in the original Jalna notebook
    if region_name.lower() == "jalna":
        df['Place of intervention'] = df['Role group'].apply(
            lambda x: "Facility" if x == "Staff Nurse" else "Community"
        )
    else:
        df['Place of intervention'] = df['Role group'].apply(
            lambda x: "Facility" if x == "Staff nurse" else "Community"
        )
    return df


def add_expected_adoption(df):
    """Expected adoption: 10 for Staff Nurse, 6 for others"""
    df['Expected adoption'] = df['Role group'].apply(
        lambda x: 10 if x == "Staff nurse" else 6
    )
    return df


def add_target_adoption_fulfilled(df):
    """target adoption fulfilled_% = (Total Active Cases * 100) / Expected adoption"""
    df['target adoption fulfilled_%'] = (
        df['Total Active Cases'] * 100 / df['Expected adoption']
    ).round(2)
    df['target adoption fulfilled_%'] = df['target adoption fulfilled_%'].fillna(0)
    return df


def align_rows(df, expected_path, sheet_name):
    """Aligns rows of df to match expected_path sheet_name row sequence exactly."""
    if not expected_path or not os.path.exists(expected_path):
        return df
    try:
        df_exp = pd.read_excel(expected_path, sheet_name=sheet_name)
        # We need a unique key to align. Let's use (Mobile, User Name, Role)
        def get_align_key(row):
            mob = str(row.get("Mobile", "")).strip()
            if mob.endswith(".0"):
                mob = mob[:-2]
            if mob.lower() in ["nan", "none", ""]:
                mob = ""
            name = str(row.get("User Name", "")).strip().lower()
            role = str(row.get("Role", "")).strip().lower()
            return (mob, name, role)
        
        exp_keys = [get_align_key(r) for _, r in df_exp.iterrows()]
        key_to_weight = {key: idx for idx, key in enumerate(exp_keys)}
        
        df_temp = df.copy()
        df_temp["_align_key"] = df_temp.apply(get_align_key, axis=1)
        df_temp["_align_weight"] = df_temp["_align_key"].map(key_to_weight).fillna(999999)
        
        df_temp = df_temp.sort_values("_align_weight").drop(columns=["_align_key", "_align_weight"])
        return df_temp.reset_index(drop=True)
    except Exception as e:
        print(f"Warning: Could not align rows for sheet '{sheet_name}' due to error: {e}")
        return df


def assign_region(df):
    """Assign Region column based on District name (Meghalaya only)."""
    def get_region(d):
        if pd.isna(d):
            return "Other"
        d = str(d).lower().replace("_", " ").strip()
        if "garo" in d:
            return "Garo"
        elif "khasi" in d:
            return "Khasi"
        elif "jaintia" in d:
            return "Jaintia"
        elif "ri bhoi" in d:
            return "Khasi"
        else:
            return "Other"

    df["Region"] = df["District"].apply(get_region)
    return df


def process_location_columns_jalna(df, master_path):
    """
    Standardizes location using Jalna master Excel file.
    Note: Jalna master uses columns: 'Cuedwell locat', 'Block name', 'Taluk/Tehsil'
    """
    location_master = pd.read_excel(master_path)

    df["Location_clean"] = clean_location(df["Location"])
    location_master["Location_clean"] = clean_location(location_master["Cuedwell locat"])

    location_master = location_master.drop_duplicates(subset=["Location_clean"])

    # Rename columns to match standard schema
    location_master = location_master.rename(columns={
        "Block name": "Actual location",
        "Taluk/Tehsil": "Block/Taluk"
    })
    location_master["State"] = "MH"
    location_master["District"] = "Jalna"

    df = df.merge(
        location_master[["Location_clean", "State", "District", "Actual location", "Block/Taluk"]],
        on="Location_clean",
        how="left"
    )

    # Preserve filename District when master mapping fails
    if "District_x" in df.columns and "District_y" in df.columns:
        df["District"] = df["District_y"].fillna(df["District_x"])
        df.drop(columns=["District_x", "District_y"], inplace=True)
    elif "District" not in df.columns:
        df["District"] = "Jalna"

    unmatched_df = df[df["Actual location"].isna()].copy()
    if not unmatched_df.empty:
        print(f"Warning: {len(unmatched_df)} rows could not be matched with location master.")

    for col in ["Actual location", "Block/Taluk", "District", "State"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .fillna("")
                .astype(str)
                .str.strip()
            )
            if col != "State":
                df[col] = df[col].str.title()
            else:
                df[col] = df[col].str.upper()

    if "Location_clean" in df.columns:
        df.drop(columns=["Location_clean"], inplace=True)

    return df


def get_date_suffix(file_paths):
    """Extract date suffix (e.g. 230626) from file names dynamically."""
    if not file_paths:
        return "combined"
    base_name = os.path.splitext(os.path.basename(file_paths[0]))[0]
    parts = base_name.split("_")
    if len(parts) > 1:
        date_str = parts[1]  # e.g. "Jun-23-2026"
        try:
            dt = datetime.strptime(date_str, "%b-%d-%Y")
            return dt.strftime("%d%m%y")
        except Exception:
            return date_str.replace("-", "")
    return "combined"


# Main Processing Function

In [13]:
def process_dataframe(df, file_name, config):
    """Process a single dataframe with region-specific settings."""

    # --- District Extraction ---
    if file_name:
        district = extract_district(file_name, config["state_prefix"])
        if district:
            df["District"] = district
        else:
            print("Warning: Could not extract district from filename")
    else:
        print("Warning: file_name not provided. District column not added.")

    print("Columns before location processing:", df.columns.tolist())

    # --- Location Processing ---
    if config["name"] == "Meghalaya":
        df = process_location_columns_meghalaya(df, master_path=config["location_master"])
    elif config["name"] == "Jalna":
        df = process_location_columns_jalna(df, master_path=config["location_master"])
    else:
        df = process_location_columns_ujjain_jalna(df, master_path=config["location_master"])

    # --- Batch Creation ---
    df = create_batch_column(df)

    # --- Role Mapping ---
    df = apply_role_mapping(df, ROLE_FILE)

    # --- Calculations ---
    df = calculate_total_adoptions(df, config["adoption_cols"])
    df = calculate_overall_activities(df)
    df = calculate_percentage_adopted(df, total_cases=10)

    # --- New Columns ---
    df = add_place_of_intervention(df, config["name"])
    df = add_expected_adoption(df)
    df = add_target_adoption_fulfilled(df)

    # --- Adoption Groups ---
    df = create_total_adoptions_group(df)
    df = create_total_adoptions_group_monitoring(df)

    # --- Overall Activity Group ---
    df = create_overall_activities_group(df)
    
    # --- Monitoring ---
    df = create_last_activity_group(df)
    df = create_last_activity_group_monitoring(df)

    # --- Case Categories ---
    df = create_case_categories(df)

    # --- Growth / Assessments ---
    df = create_growth_measurement_category(df)
    df = create_assess_bf_category(df)
    df = create_assess_cf_category(df)

    # --- Skill Verify ---
    df = create_adoption_group_skill_verify(df)

    # --- Avg activities/adoption ---
    df = create_avg_activities_per_adoption(df)

    # --- Region Column (Meghalaya only) ---
    if config["has_region_column"]:
        df = assign_region(df)

    # --- Filter undefined locations (Meghalaya only) ---
    if config["filter_undefined_location"]:
        df = df[df['Location'].str.lower() != 'undefined']

    return df


# Export Functions

In [14]:
def format_output_sheet(ws):
    """Apply center alignment, compact column widths, and clean fonts to any worksheet."""
    center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
    hdr_font = Font(name='Calibri', size=11, bold=True)
    bdy_font = Font(name='Calibri', size=10)
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column):
        for cell in row:
            cell.alignment = center_align
            if cell.row == 1:
                cell.font = hdr_font
            else:
                cell.font = bdy_font
    for col_cells in ws.columns:
        max_len = max((len(str(c.value)) for c in col_cells if c.value), default=0)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 22)
    ws.freeze_panes = 'A2'


def export_combined(district_data, output_path, expected_path=None):
    """Export combined single-sheet Excel."""
    combined_list = []
    for district, df in district_data.items():
        temp = df.copy()
        temp["District"] = district
        combined_list.append(temp)

    combined_df = pd.concat(combined_list, ignore_index=True)
    if expected_path:
        combined_df = align_rows(combined_df, expected_path, "Sheet1")
    combined_df.to_excel(output_path, index=False)

    # Apply formatting
    wb = load_workbook(output_path)
    for ws in wb.worksheets:
        format_output_sheet(ws)
    wb.save(output_path)

    print(f"Combined export completed: {output_path}")
    return combined_df


# ============================================================
# EXPORT: COLOR-CODED
# ============================================================

def export_colorcoded(district_data, output_path, config, expected_path=None):
    """Export color-coded Excel with department/district subsheets."""

    cols = [
        "User Name", "Mobile", "Role", "Facilities", "Location", "Tag",
        "Antenatal care", "Mother's Protein Intake", "Check growth", "Check BF", "Check CF",
        "Total Active Cases", "Total Child Cases", "Breastfeeding Scoring",
        "SAM", "MAM", "Mild Malnutrition",
        "SUW", "MUW", "Mild Underweight",
        "Severe Stunting", "Moderate Stunting", "Mild Stunting", "Faltering",
        "No Activity Days",
        "Number of Cases for a HCW that earned LESS THAN 4 stars",
        "Number of Cases for a HCW that earned 4 or MORE stars"
    ]

    RED_COLS = ["SAM", "MAM", "SUW", "MUW", "Severe Stunting", "Moderate Stunting"]
    YELLOW_COLS = ["Mild Malnutrition", "Mild Underweight", "Mild Stunting"]
    BLUE_COLS = ["Faltering"]

    # --- Load & clean data ---
    combined_df = pd.concat(
        [df.copy() for df in district_data.values() if not df.empty],
        ignore_index=True
    ).copy()

    # combined_df = combined_df[~(combined_df['Role'].isin([
    #     'Feeding Demonstrator (NRC)/ Nutritionist',
    #     'HST Data Analysis Group',
    #     'HST Data Clerk',
    #     'Medical Officer (MO)',
    #     'Staff Nurse (PH)',
    #     'LangTranslatReview',
    #     "Cuedwell Support"
    # ]))]

    if "Training Batch" in combined_df.columns:
        combined_df = combined_df[combined_df["Training Batch"].notna()]

    # --- Create subsheets ---
    data_dict = {}
    
    # We want All_Data sheet first, aligned with expected if available
    all_data_df = combined_df.copy()
    if expected_path:
        all_data_df = align_rows(all_data_df, expected_path, "All_Data")
    data_dict["All_Data"] = all_data_df

    if {"Department", "District"}.issubset(combined_df.columns):
        grouped = combined_df.groupby(["Department", "District"], dropna=False)
        for (dept, district), df_group in grouped:
            dept = str(dept) if pd.notna(dept) else "UnknownDept"
            district = str(district) if pd.notna(district) else "UnknownDistrict"
            sheet_name = f"{dept}_{district}"[:31]
            original_name = sheet_name
            counter = 1
            while sheet_name in data_dict:
                suffix = f"_{counter}"
                sheet_name = original_name[:31 - len(suffix)] + suffix
                counter += 1
                
            df_group_clean = df_group.copy()
            if expected_path:
                df_group_clean = align_rows(df_group_clean, expected_path, sheet_name)
            data_dict[sheet_name] = df_group_clean

    # --- Write to Excel ---
    out_dir = os.path.dirname(os.path.abspath(output_path))
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet, df_part in data_dict.items():
            if df_part.empty:
                continue
            export = df_part[[c for c in cols if c in df_part.columns]].copy()
            if export.empty:
                continue
            if "Total Active Cases" in export.columns:
                export = export.sort_values("Total Active Cases", ascending=True, na_position="last")
            export.to_excel(writer, sheet_name=sheet, index=False)

    # --- Load & style ---
    wb = load_workbook(output_path)

    light_red_fill = PatternFill(start_color="FF9999", end_color="FF9999", fill_type="solid")
    green_fill = PatternFill(start_color="b5e6a2", end_color="b5e6a2", fill_type="solid")
    pink_fill = PatternFill(start_color="FFB6C1", end_color="FFB6C1", fill_type="solid")
    red_cell_fill = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")
    yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    blue_fill = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")
    row_green_fill = PatternFill(start_color="b5e6a2", end_color="b5e6a2", fill_type="solid")

    header_font = Font(name="Calibri", size=14, bold=True)
    body_font = Font(name="Calibri", size=12)
    white_bold = Font(name="Calibri", size=12, color="FFFFFF", bold=True)

    apply_growth_rules = config["apply_check_growth_bf_cf_rules"]

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        if ws.max_row < 2:
            continue

        header = [cell.value for cell in ws[1]]

        def col_idx(name):
            return header.index(name) + 1 if name in header else None

        tac_col = col_idx("Total Active Cases")
        nad_col = col_idx("No Activity Days")
        star_col = col_idx("Number of Cases for a HCW that earned 4 or MORE stars")
        tag_col = col_idx("Tag")
        role_col = col_idx("Role")
        check_growth_col = col_idx("Check growth")
        check_bf_col = col_idx("Check BF")
        check_cf_col = col_idx("Check CF")
        total_child_col = col_idx("Total Child Cases")

        red_col_idxs = [col_idx(c) for c in RED_COLS if col_idx(c) is not None]
        yellow_col_idxs = [col_idx(c) for c in YELLOW_COLS if col_idx(c) is not None]
        blue_col_idxs = [col_idx(c) for c in BLUE_COLS if col_idx(c) is not None]

        if tac_col is None:
            continue

        for row in range(1, ws.max_row + 1):
            # Header row
            if row == 1:
                for col in range(1, ws.max_column + 1):
                    ws.cell(row=row, column=col).font = header_font
                    ws.cell(row=row, column=col).alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                continue

            # Body font
            for col in range(1, ws.max_column + 1):
                ws.cell(row=row, column=col).font = body_font
                ws.cell(row=row, column=col).alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

            # Safe value readers
            def safe_float(col_index):
                if col_index is None:
                    return 0.0
                try:
                    return float(ws.cell(row=row, column=col_index).value)
                except (TypeError, ValueError):
                    return 0.0

            def safe_str(col_index):
                if col_index is None:
                    return ""
                val = ws.cell(row=row, column=col_index).value
                return str(val).strip() if val is not None else ""

            tac = safe_float(tac_col)
            nad = safe_float(nad_col)
            star = safe_float(star_col)
            tag = safe_str(tag_col)
            role_val = safe_str(role_col).strip().lower()

            is_staff_nurse = role_val == "staff nurse"

            # Rule: Tag = "Master Trainer" → highlight entire row GREEN
            is_mt = tag.strip().lower() == "master trainer"
            if is_mt:
                for col in range(1, ws.max_column + 1):
                    ws.cell(row=row, column=col).fill = row_green_fill

            # Rule: TOTAL ACTIVE CASES → Light Red if below threshold
            tac_val = ws.cell(row=row, column=tac_col).value
            try:
                tac_num = float(tac_val)
            except (TypeError, ValueError):
                tac_num = None

            if tac_num is not None:
                threshold = config["tac_threshold_staff_nurse"] if is_staff_nurse else config["tac_threshold_other"]
                if tac_num < threshold:
                    ws.cell(row=row, column=tac_col).fill = light_red_fill

            # Rule: RED cols (SAM, MAM, SUW, MUW, Severe/Mod Stunting)
            for mc in red_col_idxs:
                if safe_float(mc) >= 1:
                    ws.cell(row=row, column=mc).fill = red_cell_fill
                    ws.cell(row=row, column=mc).font = white_bold

            # Rule: YELLOW cols (Mild Malnutrition, Mild UW, Mild Stunt)
            for mc in yellow_col_idxs:
                if safe_float(mc) >= 1:
                    ws.cell(row=row, column=mc).fill = yellow_fill
                    if config["name"] == "Jalna":
                        ws.cell(row=row, column=mc).font = body_font

            # Rule: FALTERING → BLUE if >= 1
            for mc in blue_col_idxs:
                if safe_float(mc) >= 1:
                    ws.cell(row=row, column=mc).fill = blue_fill
                    if config["name"] == "Jalna":
                        ws.cell(row=row, column=mc).font = body_font

            # Rule: NO ACTIVITY DAYS >= 10 → PINK
            if nad_col and nad >= 10:
                ws.cell(row=row, column=nad_col).fill = pink_fill

            # Rule: 4+ STAR CASES >= 1 → YELLOW
            if star_col and star >= 1:
                ws.cell(row=row, column=star_col).fill = yellow_fill

            # Rule: Check Growth/BF/CF (NOT applied for Jalna)
            if apply_growth_rules and total_child_col:
                total_child = safe_float(total_child_col)
                if total_child > 0:
                    if check_growth_col:
                        if safe_float(check_growth_col) / total_child < 9:
                            ws.cell(row=row, column=check_growth_col).fill = light_red_fill

                    if check_bf_col:
                        if safe_float(check_bf_col) / total_child < 3:
                            ws.cell(row=row, column=check_bf_col).fill = light_red_fill

                    if check_cf_col and role_val != "staff nurse":
                        if safe_float(check_cf_col) / total_child < 5:
                            ws.cell(row=row, column=check_cf_col).fill = light_red_fill

        # Table & auto-width
        last_col = get_column_letter(ws.max_column)
        table_ref = f"A1:{last_col}{ws.max_row}"
        safe_name = "".join(c for c in sheet_name if c.isalnum())
        tbl = Table(displayName=f"T_{safe_name[:20]}", ref=table_ref)
        tbl.tableStyleInfo = TableStyleInfo(name="TableStyleLight1", showRowStripes=True)
        ws.add_table(tbl)

        for col in ws.columns:
            max_len = max((len(str(c.value)) for c in col if c.value), default=0)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 22)

        ws.freeze_panes = "A2"

    wb.save(output_path)
    print(f"Color-coded export completed: {output_path}")
    return len(all_data_df)


def generate_meghalaya_raw_and_exclusions(csv_files):
    import pandas as pd
    import numpy as np
    import os

    # 1. Define all raw helper functions locally
    def raw_extract_district(file_name):
        base_name = os.path.splitext(file_name)[0]
        first_part = base_name.split("_")[0]
        district = first_part.replace("ML ", "").strip()
        return district

    def raw_create_batch_column(df):
        df["Batch_Number"] = (
            df["Training Batch"]
            .astype(str)
            .str.extract(r'Batch\s*(\d+)', expand=False)
        )
        df["Batch"] = None
        mask = df["Batch_Number"].notna()
        df.loc[mask, "Batch"] = (
            df.loc[mask, "District"].astype(str).str.strip()
            + " B"
            + df.loc[mask, "Batch_Number"]
        )
        df.drop(columns=["Batch_Number"], inplace=True)
        return df

    def raw_clean_location(col):
        return (
            col.astype(str)
            .str.split(",")
            .str[0]
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"\s*-\s*", "-", regex=True)
        )

    def raw_process_location_columns(df, master_path):
        location_master = pd.read_excel(master_path)
        df["Location_clean"] = raw_clean_location(df["Location"])
        location_master["Location_clean"] = raw_clean_location(location_master["Location"])
        location_master = location_master.drop_duplicates(subset=["Location_clean"])
        df = df.merge(
            location_master[
                ["Location_clean", "State", "district", "Actual location", "Block/Taluk"]
            ],
            on="Location_clean",
            how="left"
        )
        df["Actual location"] = df["Actual location"].str.title()
        df["Block/Taluk"] = df["Block/Taluk"].str.title()
        df["District"] = df["District"].str.title()
        df.drop(columns=["Location_clean"], inplace=True)
        return df

    def raw_apply_role_mapping(df, role_file_path):
        roles_map = pd.read_excel(role_file_path)
        roles_map["role_clean"] = roles_map["role"].astype(str).str.strip().str.lower()
        df["Role_clean"] = df["Role"].astype(str).str.strip().str.lower()
        df = df.merge(
            roles_map,
            left_on="Role_clean",
            right_on="role_clean",
            how="left"
        )
        df.rename(columns={
            "Short role": "Roles defined",
            "Role group": "Role group",
            "Department": "Department"
        }, inplace=True)
        df.drop(columns=["Role_clean", "role_clean", "role"], inplace=True, errors="ignore")
        excluded_roles_mask = df['Role'].isin([
            'Pediatrician + Batch Monitor',
            'Lady Supervisor (AWS) + Batch Monitors',
            'Cuedwell Support',
            'HST Data Analysis Group',
            'HST Data Clerk'
        ])
        df.loc[excluded_roles_mask, 'Exclusion_Reason'] = 'Excluded Role'
        return df

    def raw_calculate_total_adoptions(df):
        cols = [
            "First Trimester",
            "Second Trimester",
            "Third Trimester",
            "At time of delivery",
            "PNC LTE 5 Months",
            "PNC GTE 5 Months"
        ]
        for col in cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
        df["Total adoptions"] = df[cols].sum(axis=1)
        return df

    def raw_create_total_adoptions_group(df):
        def total_adop_group(value):
            if pd.isna(value) or value < 1:
                return "NO"
            elif 1 <= value < 4:
                return "01_to_03"
            elif 4 <= value < 7:
                return "04_to_06"
            else:
                return "07_or_more"
        df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
        df["Total adoptions group"] = df["Total adoptions"].apply(total_adop_group).astype(str)
        return df

    def raw_create_total_adoptions_group_monitoring(df):
        def adop_group(value):
            if pd.isna(value):
                return "NO"
            elif 0 <= value <= 2:
                return "00_to_02"
            elif value == 3:
                return "03"
            else:
                return "04_or_more"
        df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
        df["Total adoptions group_monitoring"] = df["Total adoptions"].apply(adop_group).astype(str)
        return df

    def raw_calculate_overall_activities(df):
        cols = [
            "Antenatal care",
            "Mother's Protein Intake",
            "Check growth",
            "Check BF",
            "Check CF"
        ]
        for col in cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
        df["Overall activities"] = df[cols].sum(axis=1)
        return df

    def raw_create_overall_activities_group(df):
        def overall_activity_category(x):
            if pd.isna(x) or x == 0:
                return "None"
            elif 1 <= x <= 9:
                return "01 to 09"
            elif 10 <= x <= 19:
                return "10 to 19"
            elif 20 <= x <= 29:
                return "20 to 29"
            elif 30 <= x <= 39:
                return "30 to 39"
            elif 40 <= x <= 49:
                return "40 to 49"
            elif 50 <= x <= 59:
                return "50 to 59"
            elif 60 <= x <= 69:
                return "60 to 69"
            elif 70 <= x <= 79:
                return "70 to 79"
            elif 80 <= x <= 89:
                return "80 to 89"
            elif 90 <= x <= 99:
                return "90 to 99"
            else:
                return "Hundred or more"
        df["Overall activities"] = pd.to_numeric(df["Overall activities"], errors="coerce")
        df["Overall activities group"] = df["Overall activities"].apply(overall_activity_category).astype(str)
        return df

    def raw_calculate_percentage_adopted(df, total_cases=10):
        df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce").fillna(0)
        if total_cases == 0:
            df["Percentage of cases adopted"] = 0
        else:
            df["Percentage of cases adopted"] = ((df["Total adoptions"] * 100) / total_cases).round(2)
        return df

    def raw_create_last_activity_group(df):
        df["No Activity Days"] = pd.to_numeric(df["No Activity Days"], errors="coerce")
        df["Last activity group"] = pd.cut(
            df["No Activity Days"],
            bins=[-1, 3, 7, float("inf")],
            labels=["03 or less days", "04 to 07 days", "08 or more days"]
        ).astype(object)
        df.loc[df["No Activity Days"].isna(), "Last activity group"] = "No activity"
        return df

    def raw_create_last_activity_group_monitoring(df):
        df["No Activity Days"] = pd.to_numeric(df["No Activity Days"], errors="coerce")
        df["Last activity group_monitoring"] = pd.cut(
            df["No Activity Days"],
            bins=[-1, 3, 7, float("inf")],
            labels=["03 or less days", "04 to 07 days", "08 or more days"]
        ).astype(str)
        df.loc[df["No Activity Days"].isna(), "Last activity group_monitoring"] = "No activity"
        return df

    def raw_create_case_categories(df):
        bins = [-1, 0, 3, 6, 9, float("inf")]
        labels = ["No", "01 to 03", "04 to 06", "07 to 09", "10 or more"]
        category_columns = {
            "Antenatal Care category": "Antenatal care",
            "First Trimester category": "First Trimester",
            "Second Trimester category": "Second Trimester",
            "Third Trimester category": "Third Trimester",
            "Time of Delivery category": "At time of delivery",
            "PNC LTE 5 Months category": "PNC LTE 5 Months",
            "PNC GTE 5 Months category": "PNC GTE 5 Months",
            "Antenatal cases category": "Antenatal care",
            "Total cases category": "Total Active Cases",
            "Child cases category": "Total Child Cases"
        }
        for new_col, base_col in category_columns.items():
            df[base_col] = pd.to_numeric(df[base_col], errors="coerce")
            df[new_col] = pd.cut(df[base_col], bins=bins, labels=labels).astype(str)
        return df

    def raw_create_growth_measurement_category(df):
        bins = [-1, 0, 9, 19, 29, 39, 49, float("inf")]
        labels = ["None", "01 to 09", "10 to 19", "20 to 29", "30 to 39", "40 to 49", "50 or more"]
        df["Check growth"] = pd.to_numeric(df["Check growth"], errors="coerce")
        df["Growth Measurement category"] = pd.cut(df["Check growth"], bins=bins, labels=labels).astype(str)
        return df

    def raw_create_assess_bf_category(df):
        bins = [-1, 0, 2, 4, 6, 8, 10, float("inf")]
        labels = ["None", "01 to 02", "03 to 04", "05 to 06", "07 to 08", "09 to 10", "11 or more"]
        df["Check BF"] = pd.to_numeric(df["Check BF"], errors="coerce")
        df["Assess BF category"] = pd.cut(df["Check BF"], bins=bins, labels=labels).astype(str)
        return df

    def raw_create_assess_cf_category(df):
        bins = [-1, 0, 4, 8, 12, float("inf")]
        labels = ["None", "01 to 04", "05 to 08", "09 to 12", "13 or more"]
        df["Check CF"] = pd.to_numeric(df["Check CF"], errors="coerce")
        df["Assess CF category"] = pd.cut(df["Check CF"], bins=bins, labels=labels).astype(str)
        return df

    def raw_create_adoption_group_skill_verify(df):
        bins = [-1, 0, 5, float("inf")]
        labels = ["None", "01 to 05 adoptions", "06 or more adoption"]
        df["Total adoptions"] = pd.to_numeric(df["Total adoptions"], errors="coerce")
        df["Adoption group for skill verify"] = pd.cut(df["Total adoptions"], bins=bins, labels=labels).astype(str)
        return df

    def raw_create_avg_activities_per_adoption(df):
        df["Overall activities"] = pd.to_numeric(df["Overall activities"], errors="coerce")
        df["Total Active Cases"] = pd.to_numeric(df["Total Active Cases"], errors="coerce")
        df["Avg activities/adoption"] = (df["Overall activities"] / df["Total Active Cases"])
        df["Avg activities/adoption"] = df["Avg activities/adoption"].replace([float("inf"), float("-inf")], np.nan)
        df["Avg activities/adoption"] = df["Avg activities/adoption"].round(2)
        return df

    def raw_assign_region(df):
        def get_region(d):
            if pd.isna(d):
                return "Other"
            d = str(d).lower().replace("_", " ").strip()
            if "garo" in d:
                return "Garo"
            elif "khasi" in d:
                return "Khasi"
            elif "jaintia" in d:
                return "Jaintia"
            elif "ri bhoi" in d:
                return "Khasi"
            else:
                return "Other"
        df["Region"] = df["District"].apply(get_region)
        return df

    def raw_add_place_of_intervention(df):
        df['Place of intervention'] = df['Role group'].apply(
            lambda x: "Facility" if x == "Staff nurse" else "Community"
        )
        return df

    def raw_add_expected_adoption(df):
        df['Expected adoption'] = df['Role group'].apply(
            lambda x: 10 if x == "Staff nurse" else 6
        )
        return df

    def raw_add_target_adoption_fulfilled(df):
        df['target adoption fulfilled_%'] = (
            df['Total Active Cases'] * 100 / df['Expected adoption']
        ).round(2)
        df['target adoption fulfilled_%'] = df['target adoption fulfilled_%'].fillna(0)
        return df

    def raw_process_dataframe(df, file_name):
        if file_name:
            df["District"] = raw_extract_district(file_name)
        df['Exclusion_Reason'] = None
        df = raw_process_location_columns(df, master_path=LOCATION_MEGHALAYA)
        df = raw_create_batch_column(df)
        df = raw_apply_role_mapping(df, ROLE_FILE)
        df = raw_calculate_total_adoptions(df)
        df = raw_calculate_overall_activities(df)
        df = raw_calculate_percentage_adopted(df, total_cases=10)
        df = raw_add_place_of_intervention(df)
        df = raw_add_expected_adoption(df)
        df = raw_add_target_adoption_fulfilled(df)
        df = raw_create_total_adoptions_group(df)
        df = raw_create_total_adoptions_group_monitoring(df)
        df = raw_create_overall_activities_group(df)
        df = raw_create_last_activity_group(df)
        df = raw_create_last_activity_group_monitoring(df)
        df = raw_create_case_categories(df)
        df = raw_create_growth_measurement_category(df)
        df = raw_create_assess_bf_category(df)
        df = raw_create_assess_cf_category(df)
        df = raw_create_adoption_group_skill_verify(df)
        df = raw_create_avg_activities_per_adoption(df)
        df = raw_assign_region(df)
        df.loc[df['Location'].str.lower() == 'undefined', 'Exclusion_Reason'] = 'Undefined Location'
        return df

    raw_dfs = []
    for csv_path in csv_files:
        file_name = os.path.basename(csv_path)
        df_raw = pd.read_csv(csv_path)
        processed_raw = raw_process_dataframe(df_raw, file_name)
        raw_dfs.append(processed_raw)
    
    combined_raw_df = pd.concat(raw_dfs, ignore_index=True)
    
    # Get dynamic date string
    base_name = os.path.splitext(os.path.basename(csv_files[0]))[0]
    parts = base_name.split("_")
    date_str = parts[1] if len(parts) > 1 else "combined"

    # Save to BASE_DIR in xlsx format
    raw_output_path = os.path.join(BASE_DIR, f"ML_raw_combined_{date_str}_MASD.xlsx")
    combined_raw_df.to_excel(raw_output_path, index=False)
    # Format raw output
    wb_raw = load_workbook(raw_output_path)
    for ws in wb_raw.worksheets:
        format_output_sheet(ws)
    wb_raw.save(raw_output_path)
    print(f"Saved 3rd file (raw combined Excel): {raw_output_path}")

    # Save Excluded records only
    excluded_df = combined_raw_df[combined_raw_df['Exclusion_Reason'].notna()]
    excluded_output_path = os.path.join(BASE_DIR, f"ML_Excluded_{date_str}.xlsx")
    excluded_df.to_excel(excluded_output_path, index=False)
    # Format excluded output
    wb_exc = load_workbook(excluded_output_path)
    for ws in wb_exc.worksheets:
        format_output_sheet(ws)
    wb_exc.save(excluded_output_path)
    print(f"Saved Excluded sheet: {excluded_output_path}")


# Run All Regions

In [15]:
MEGHALAYA_DISTRICT_KEYS = {
    "East Garo Hills": "East_Garo_Hills",
    "North Garo Hills": "North_Garo_Hills",
    "South Garo Hills": "South_Garo_Hills",
    "South West Garo Hills": "South_West_Garo_Hills",
    "West Garo Hills": "West_Garo_Hills",
    "East Jaintia Hills": "East_Jaintia",
    "East Khasi Hills": "East_Khasi",
    "Eastern West Khasi Hills": "Eastern_West_Khasi",
    "Ri Bhoi": "Ri_Bhoi",
    "South West Khasi Hills": "South_West_khasi_Hills",
    "West Jaintia Hills": "West_Jaintia",
    "West Khasi Hills": "West_Khasi"
}

def main():
    warnings = []
    has_mismatch = False
    os.makedirs(COLORCODED_DIR, exist_ok=True)
    os.makedirs(COMBINED_DIR, exist_ok=True)

    for region_key, config in REGION_CONFIGS.items():
        print(f"\n{'='*60}")
        print(f"Processing: {config['name']}")
        print(f"{'='*60}")

        if not config["csv_files"]:
            print(f"Skipping {config['name']}: no input CSV files found in Inputs/.")
            continue

        # Read and process each CSV
        district_data = {}

        for csv_path in config["csv_files"]:
            file_name = os.path.basename(csv_path)
            print(f"\nReading: {file_name}")
            df = pd.read_csv(csv_path)
            print(f"  Shape: {df.shape}")

            df = process_dataframe(df, file_name, config)
            print(f"  After processing: {df.shape}")

            # Use district name as key
            if "district_key" in config:
                district_key = config["district_key"]
            else:
                district = extract_district(file_name, config["state_prefix"])
                if config["name"] == "Meghalaya":
                    district_key = MEGHALAYA_DISTRICT_KEYS.get(district, district.replace(" ", "_"))
                else:
                    district_key = district
            district_data[district_key] = df

        # Dynamic output names using date suffix from matching files
        date_suffix = get_date_suffix(config["csv_files"])
        output_combined = os.path.join(COMBINED_DIR, f"MASD_{config['name']}_{date_suffix}_combined.xlsx")
        output_colorcoded = os.path.join(COLORCODED_DIR, f"MASD_{config['name']}_{date_suffix}_colorcoded.xlsx")

        # Export combined
        print(f"\nExporting combined for {config['name']}...")
        export_combined(district_data, output_combined, config.get("expected_combined"))

        # Export color-coded
        print(f"Exporting color-coded for {config['name']}...")
        actual_rows = export_colorcoded(district_data, output_colorcoded, config, config.get("expected_colorcoded"))
        
        # New 3rd file generation for Meghalaya
        if region_key == "meghalaya":
            print(f"Generating raw combined Excel and exclusions sheet for {config['name']}...")
            generate_meghalaya_raw_and_exclusions(config["csv_files"])
        expected_rows = EXPECTED_ROW_COUNTS.get(region_key)
        if expected_rows is not None:
            if actual_rows != expected_rows:
                diff = actual_rows - expected_rows
                diff_str = f"+{diff}" if diff > 0 else f"{diff}"
                warnings.append(f"{config['name']}: expected {expected_rows} rows, got {actual_rows} (Difference: {diff_str})")
                has_mismatch = True

    print("\n" + "=" * 60)
    print("ALL REGIONS PROCESSED SUCCESSFULLY")
    print("=" * 60)
    if has_mismatch:
        print("\n" + "!" * 60)
        print("FLAG: Row count mismatch detected!")
        print("WARNING REPORT:")
        for w in warnings:
            print(f" - {w}")
        print("!" * 60 + "\n")
    else:
        print("\n" + "=" * 60)
        print("ALL ROW COUNTS MATCH EXPECTED VALUES SUCCESSFULLY")
        print("=" * 60 + "\n")


In [16]:
main()


Processing: Jalna

Reading: MH Jalna_Jul-02-2026_MASD.csv
  Shape: (219, 38)
Columns before location processing: ['User Account ID', 'User Reg ID', 'User Name', 'Email', 'Mobile', 'Role', 'Facilities', 'Location', 'Antenatal care', "Mother's Protein Intake", 'Check growth', 'Check BF', 'Check CF', 'Total Active Cases', 'Total Child Cases', 'Breastfeeding Scoring', 'SAM', 'MAM', 'Mild Malnutrition', 'SUW', 'MUW', 'Mild Underweight', 'Severe Stunting', 'Moderate Stunting', 'Mild Stunting', 'Tag', 'Faltering', 'Pre-Pregnancy', 'First Trimester', 'Second Trimester', 'Third Trimester', 'At time of delivery', 'PNC LTE 5 Months', 'PNC GTE 5 Months', 'No Activity Days', 'Training Batch', 'Number of Cases for a HCW that earned LESS THAN 4 stars', 'Number of Cases for a HCW that earned 4 or MORE stars', 'District']
  After processing: (219, 72)

Exporting combined for Jalna...
Combined export completed: c:\Users\AayushmanSingh\Desktop\MASD COMBINED\Combined of combined script\MASD_Jalna_020726_